


#Import Libraries


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt
import seaborn as sns

#Load Dataset

In [7]:
df = pd.read_csv("diabetes.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


#Explore Dataset

In [ ]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [ ]:
df.describe()



,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [ ]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


#Select Features and Target

In [11]:
X = df.drop("Outcome", axis=1)

y = df["Outcome"]


#Train-Test Split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Baseline Model

In [14]:
baseline_model = RandomForestClassifier(random_state=42)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

#Evaluate Baseline Model

In [15]:
baseline_accuracy = accuracy_score(y_test, baseline_pred)

baseline_precision = precision_score(y_test, baseline_pred)

baseline_recall = recall_score(y_test, baseline_pred)

baseline_f1 = f1_score(y_test, baseline_pred)

baseline_auc = roc_auc_score(y_test, baseline_pred)

print("Accuracy:", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall:", baseline_recall)
print("F1 Score:", baseline_f1)
print("ROC AUC:", baseline_auc)

Accuracy: 0.7207792207792207
Precision: 0.6071428571428571
Recall: 0.6181818181818182
F1 Score: 0.6126126126126126
ROC AUC: 0.697979797979798


#Grid Search Hyperparameter Tuning

In [17]:
param_grid = {
    'n_estimators': [50,100,200],
    'max_depth': [3,5,10,None],
    'min_samples_split': [2,5,10]
}

In [18]:
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [3, 5, 10, None],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy')

#Best Parameters

In [19]:
print(grid_search.best_params_)

print(grid_search.best_score_)

{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
0.7834199653471945


#Build Optimized Model

In [20]:
best_model = grid_search.best_estimator_

best_pred = best_model.predict(X_test)

#Evaluate Tuned Model

In [21]:
accuracy = accuracy_score(y_test,best_pred)

precision = precision_score(y_test,best_pred)

recall = recall_score(y_test,best_pred)

f1 = f1_score(y_test,best_pred)

auc = roc_auc_score(y_test,best_pred)

print("Accuracy:",accuracy)
print("Precision:",precision)
print("Recall:",recall)
print("F1 Score:",f1)
print("ROC AUC:",auc)

Accuracy: 0.7337662337662337
Precision: 0.6206896551724138
Recall: 0.6545454545454545
F1 Score: 0.6371681415929203
ROC AUC: 0.7161616161616162


#Cross Validation

In [22]:
cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print(cv_scores)

print("Mean Accuracy:",cv_scores.mean())

[0.77272727 0.71428571 0.76623377 0.83006536 0.74509804]
Mean Accuracy: 0.7656820303879128


#Comparison Table

In [23]:
comparison = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Before Tuning":[
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_auc
    ],
    "After Tuning":[
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]
})

comparison

,Metric,Before Tuning,After Tuning
0,Accuracy,0.720779,0.733766
1,Precision,0.607143,0.620690
2,Recall,0.618182,0.654545
3,F1 Score,0.612613,0.637168
4,ROC-AUC,0.697980,0.716162


#key Insight
Hyperparameter tuning improved model performance

The Random Forest model showed improvement after applying GridSearchCV. Accuracy increased from 72.08% to 73.38%, demonstrating that selecting optimal hyperparameters helped the model make better predictions.

Improved ability to identify diabetic patients

Recall increased from 61.82% to 65.45%. This is particularly important in healthcare applications because it means the model became better at identifying individuals who have diabetes, reducing the number of missed positive cases.

Better balance between Precision and Recall

The F1-Score increased from 61.26% to 63.72%. This indicates that the tuned model achieved a better balance between correctly identifying diabetic patients and minimizing false predictions.

Improved overall discrimination capability

ROC-AUC improved from 69.80% to 71.62%. This suggests that the optimized model became more effective at distinguishing between diabetic and non-diabetic individuals across different classification thresholds.

Cross-validation provided more reliable evaluation

Using K-Fold Cross-Validation helped ensure that the model's performance was not dependent on a single train-test split. This increased confidence that the model can generalize reasonably well to unseen data.

#Conclusion

The objective of this project was to improve and validate a machine learning model for diabetes prediction using hyperparameter tuning and cross-validation techniques. A baseline Random Forest model was first developed and evaluated. GridSearchCV was then applied to identify the optimal combination of hyperparameters.

The tuned model outperformed the baseline model across all evaluation metrics, including Accuracy, Precision, Recall, F1-Score, and ROC-AUC. Although the performance improvements were moderate, the results demonstrate the value of model tuning in enhancing predictive accuracy and reliability.

The project also highlighted the importance of cross-validation in obtaining a more robust estimate of model performance and reducing the risk of overfitting.

#Recommendations

**Collect More Data**

The Pima Indians Diabetes dataset contains a limited number of observations. A larger dataset could improve the model's ability to learn complex patterns and achieve higher predictive performance.

**Feature Engineering**

Additional feature engineering techniques, such as creating interaction features or scaling specific variables, may further improve model performance.

**Experiment with Advanced Algorithms**

Future work could explore algorithms such as:

Gradient Boosting

XGBoost

LightGBM

CatBoost

These models often achieve better results on tabular classification datasets.

**Address Class Imbalance**

If class imbalance exists, techniques such as SMOTE, oversampling, or class weighting can be applied to improve the model's ability to detect diabetic cases.

**Deploy and Monitor the Model**

Before real-world use, the model should be deployed in a controlled environment and continuously monitored to ensure consistent performance on new data.

#Final Statement

The optimized Random Forest model achieved better predictive performance than the baseline model, demonstrating the effectiveness of hyperparameter tuning and cross-validation. The project reinforced the importance of model optimization techniques in building reliable and generalizable machine learning solutions for healthcare prediction tasks.